In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, TimeSeriesSplit, ParameterGrid
from sklearn.metrics import make_scorer
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
import catboost as cat
import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_parquet('../data/train_team_track.parquet')
test_df = pd.read_parquet('../data/test_team_track.parquet')

In [3]:
int_cols = ['route_id', 'office_from_id'] + [f'status_{i}' for i in range(1,9)]
for col in int_cols:
    train_df[col] = train_df[col].astype('int32')

train_df['target_2h'] = train_df['target_2h'].astype('float32')

In [4]:
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])

train_df = train_df.sort_values(['route_id', 'timestamp']).reset_index(drop=True)
test_df = test_df.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

In [5]:
train_df['hour'] = train_df['timestamp'].dt.hour
test_df['hour'] = test_df['timestamp'].dt.hour

train_df['weekday'] = train_df['timestamp'].dt.weekday
test_df['weekday'] = test_df['timestamp'].dt.weekday

train_df['month'] = train_df['timestamp'].dt.month
test_df['month'] = test_df['timestamp'].dt.month

In [ ]:
statuses = []
for i in range(1, 9):
    statuses.append(f'status_{i}')
lags = [1, 2, 4, 8, 12, 24, 48]
for status in statuses:
    for lag in lags:
        train_df[f'{status}_lag{lag}'] = train_df.groupby('route_id')[status].shift(lag)

for lag in lags:
    train_df[f'target_lag{lag}'] = train_df.groupby('route_id')['target_2h'].shift(lag)

max_lag = 48
train_df['row_num'] = train_df.groupby('route_id').cumcount()
# train_df['target_diff'] = train_df.groupby('route_id')['target_2h'].diff(1)

train_df = train_df[train_df['row_num'] >= max_lag].drop(columns=['row_num']).reset_index(drop=True)

MemoryError: Unable to allocate 33.1 MiB for an array with shape (4342000, 1) and data type float64

In [7]:
cols_to_delete = ['target_2h', 'timestamp', 'target_diff']
features = [col for col in train_df.columns if col not in cols_to_delete]
cat_cols = ['route_id']
cat_indices = [features.index(col) for col in cat_cols if col in features]
print(f'Number of features: {len(features)}')

Number of features: 76


In [21]:
with open('../models/features.json', 'w') as f:
    json.dump(features, f)

with open('../models/cat_indices.json', 'w') as f:
    json.dump(cat_indices, f)

In [ ]:
features

['office_from_id',
 'route_id',
 'status_1',
 'status_2',
 'status_3',
 'status_4',
 'status_5',
 'status_6',
 'status_7',
 'status_8',
 'hour',
 'weekday',
 'month',
 'status_1_lag1',
 'status_1_lag2',
 'status_1_lag4',
 'status_1_lag8',
 'status_1_lag12',
 'status_1_lag24',
 'status_1_lag48',
 'status_2_lag1',
 'status_2_lag2',
 'status_2_lag4',
 'status_2_lag8',
 'status_2_lag12',
 'status_2_lag24',
 'status_2_lag48',
 'status_3_lag1',
 'status_3_lag2',
 'status_3_lag4',
 'status_3_lag8',
 'status_3_lag12',
 'status_3_lag24',
 'status_3_lag48',
 'status_4_lag1',
 'status_4_lag2',
 'status_4_lag4',
 'status_4_lag8',
 'status_4_lag12',
 'status_4_lag24',
 'status_4_lag48',
 'status_5_lag1',
 'status_5_lag2',
 'status_5_lag4',
 'status_5_lag8',
 'status_5_lag12',
 'status_5_lag24',
 'status_5_lag48',
 'status_6_lag1',
 'status_6_lag2',
 'status_6_lag4',
 'status_6_lag8',
 'status_6_lag12',
 'status_6_lag24',
 'status_6_lag48',
 'status_7_lag1',
 'status_7_lag2',
 'status_7_lag4',
 'sta

In [9]:
last14 = train_df[train_df['timestamp'] >= (train_df['timestamp'].max()-pd.Timedelta(days=14))].copy()

In [10]:
split_idx14 = int(0.8 * len(last14))
split_idx = int(0.8 * len(train_df))

last14_train = last14.iloc[:split_idx14]
last14_val = last14.iloc[split_idx14:]

train_df1 = train_df.iloc[:split_idx]
val_df = train_df.iloc[split_idx:]

print(f'Train size (last 14 days): {len(last14_train)}')
print(f'Validation size (last 14 days): {len(last14_val)}\n')
print(f'Train size (all days): {len(train_df1)}')
print(f'Validation size (all days): {len(val_df)}')

Train size (last 14 days): 538400
Validation size (last 14 days): 134600

Train size (all days): 3435200
Validation size (all days): 858800


In [11]:
X_train14, y_train14 = last14_train[features], last14_train['target_2h']
X_val14, y_val14 = last14_val[features], last14_val['target_2h']

X_train, y_train = train_df1[features], train_df1['target_2h']
X_val, y_val = val_df[features], val_df['target_2h']

In [12]:
X_train.dtypes

office_from_id      int32
route_id            int32
status_1            int32
status_2            int32
status_3            int32
                   ...   
target_lag4       float32
target_lag8       float32
target_lag12      float32
target_lag24      float32
target_lag48      float32
Length: 76, dtype: object

In [33]:
def wape_and_bias_lgb(y_true, y_pred):
    wape = (np.abs(y_pred - y_true)).sum() / y_true.sum()
    rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
    return 'wape_rbias', wape + rbias, False

def wape_and_bias_xgb(y_true, y_pred):
    wape = (np.abs(y_pred - y_true)).sum() / y_true.sum()
    rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
    return 'wape_rbias', wape + rbias

def wape_and_bias_cat(y_true, y_pred):
    wape = (np.abs(y_pred - y_true)).sum() / y_true.sum()
    rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
    return wape + rbias

In [ ]:
'''
lgb_model = lgb.LGBMRegressor(subsample_freq=1, random_state=101, categorical_feature=cat_indices)

lgb_params = {
    'num_leaves': [31, 63, 127, 159],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [500, 1000, 1500],
    'min_child_samples': [30, 50, 100],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0.0, 0.1, 1.0, 10.0],
    'reg_lambda': [0.0, 0.1, 1.0, 10.0]
}

tss = TimeSeriesSplit(n_splits=3)

rand_search = RandomizedSearchCV(
    lgb_model,
    lgb_params,
    n_iter=15,
    scoring=scorer,
    cv=tss,
    random_state=101
)
'''
lgb_model = lgb.LGBMRegressor(
    num_leaves=63,
    learning_rate=0.03,
    n_estimators=1000,
    min_child_samples=100,
    subsample=0.7,
    colsample_bytree=0.7,
    subsample_freq=1,
    reg_lambda=0.5,
    reg_alpha=0.5,
    random_state=101,
    verbose=-1,
    categorical_feature=cat_indices
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric=wape_and_bias_lgb,
    callbacks=[lgb.early_stopping(50)]
)

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.7,
    colsample_bytree=0.7,
    early_stopping_rounds=50,
    random_state=101,
    eval_metric='mae'
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

cat_model = cat.CatBoostRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    depth=6,
    subsample=0.7,
    random_seed=101,
    early_stopping_rounds=50,
    eval_metric='MAE',
    verbose=0
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    verbose=False
)

joblib.dump(lgb_model, '../models/lgb_model.pkl')
joblib.dump(xgb_model, '../models/xgb_model.pkl')
joblib.dump(cat_model, '../models/cat_model.pkl')

# rand_search.fit(X_train, y_train)
# best_params = rand_search.best_params_
'''
final_model = lgb.LGBMRegressor(
    **best_params, subsample_freq=1, random_state=101, categorical_feature=cat_indices
)
'''

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[174]	valid_0's l2: 581.187	valid_0's wape_rbias: 0.223072


In [35]:
lgb_val = lgb_model.predict(X_val)
xgb_val = xgb_model.predict(X_val)
cat_val = cat_model.predict(X_val)

X_meta_val = np.column_stack([lgb_val, xgb_val, cat_val])

meta_model = Ridge(alpha=10.0)
meta_model.fit(X_meta_val, y_val)

max_lag = 48
history = {}
for route, group in train_df.groupby('route_id'):
    history[route] = group.tail(max_lag).copy()

predictions = []
for route, group in test_df.groupby('route_id'):
    hist = history[route].copy()
    
    for idx, row in group.iterrows():
        features_dict = hist.iloc[-1].to_dict()
        features_dict.pop('timestamp', None)
        features_dict.pop('target_2h', None)
        features_dict['hour'] = row['hour']
        features_dict['weekday'] = row['weekday']
        features_dict['month'] = row['month']

        for col in features:
            if col.startswith('status_') and '_lag' in col:
                status, str_lag = col.split('_lag')
                lag = int(str_lag)

                if lag <= len(hist):
                    features_dict[col] = hist.iloc[-lag][status]
                else:
                    features_dict[col] = hist.iloc[0][status]
        
        for col in features:
            if 'target_lag' in col:
                targ_lag = int(col.split('target_lag')[-1])
                
                if targ_lag <= len(hist):
                    features_dict[col] = hist.iloc[-targ_lag]['target_2h']
                else:
                    features_dict[col] = hist.iloc[0]['target_2h']
    
        
        X_row = pd.DataFrame([features_dict])[features]
        lgb_pred = lgb_model.predict(X_row)[0]
        xgb_pred = xgb_model.predict(X_row)[0]
        cat_pred = cat_model.predict(X_row)[0]
        X_meta_test = np.column_stack([lgb_pred, xgb_pred, cat_pred])
        pred_ensemble = meta_model.predict(X_meta_test)[0]
        predictions.append({'id': row['id'], 'y_pred': pred_ensemble})

        new_row = hist.iloc[-1:].copy()
        new_row['timestamp'] = row['timestamp']
        new_row['target_2h'] = pred_ensemble
        hist = pd.concat([hist, new_row], ignore_index=True).tail(max_lag)


submission = pd.DataFrame(predictions)
submission.to_csv('submission.csv', index=False)

In [36]:
joblib.dump(meta_model, '../models/meta_model.pkl')

['../models/meta_model.pkl']